<a href="https://colab.research.google.com/github/Adhiaris/Scikit-Learn-Cookbook/blob/main/2_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2: Pre-Model Workflow and Data Preprocessing

Data quality dictates machine learning success. Flawed data like missing values, mixed scales, or unencoded categories will directly degrade model performance.

This chapter covers critical preprocessing workflows:

- **Handling missing data:** `SimpleImputer()`, `KNNImputer()`, `IterativeImputer()`
- **Scaling features:** `StandardScaler()`, `MinMaxScaler()`, `Normalizer()`
- **Encoding categories:** `OneHotEncoder()`, `LabelEncoder()`, `ColumnTransformer()`
- **Building pipelines:** `Pipeline()` for reproducible, leak-free execution
- **Feature engineering:** `PolynomialFeatures()`, `KBinsDiscretizer()`, `RFE()`, `SelectFromModel()`

Data preprocessing requires robust, dynamic pipelines because real-world data constantly shifts.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version:        {np.__version__}")
print(f"pandas version:       {pd.__version__}")

scikit-learn version: 1.6.1
NumPy version:        2.0.2
pandas version:       2.2.2


This environment runs scikit-learn 1.8.0, NumPy 2.4.2, and pandas 3.0.1. The code is structured to execute seamlessly on Google Colab.

## The Impact of Raw Data

Models learn directly from data patterns. Poor data yields poor generalization. Common roadblocks include:

1. **Missing data:** Breaks most estimators. Requires active imputation.
2. **Outliers:** Heavily distorts linear models and distance-based metrics.
3. **Categorical variables:** Must be numericized before fitting algorithms.
4. **Feature scaling:** Disparate numeric scales skew gradient and distance calculations.
5. **Data leakage:** Accidentally fitting preprocessors on test data destroys evaluation validity.

scikit-learn offers native solutions for all these problems.

## Handling Missing Data

Missing values are universal. Three primary imputation approaches are available in scikit-learn:

| Method | Strategy | Best For |
|:---|:---|:---|
| `SimpleImputer()` | Global mean/median/mode | Fast baselines (<5% missing) |
| `KNNImputer()` | Nearest neighbors averaging | Leveraging local structural patterns |
| `IterativeImputer()` | Regression chains | Complex, multivariate dependencies |

### Creating a Dataset with Missing Values

We start by simulating a dataset featuring ~20% random NaNs.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(2024)
n_samples = 20
n_features = 10

data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}
df = pd.DataFrame(data)

for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isna().sum().sum()
print(f"Dataset shape: {df.shape[0]} samples x {df.shape[1]} features")
print(f"Total cells: {total_cells}")
print(f"Missing cells: {int(missing_cells)} ({missing_cells/total_cells*100:.1f}%)")
print(f"Missing per feature:")
print(df.isna().sum().to_string())
print()
print("First 6 rows:")
print(df.head(6).to_string())

Dataset shape: 20 samples x 10 features
Total cells: 200
Missing cells: 44 (22.0%)
Missing per feature:
Feature1     3
Feature2     5
Feature3     4
Feature4     4
Feature5     5
Feature6     5
Feature7     7
Feature8     6
Feature9     2
Feature10    3

First 6 rows:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015       NaN   42.0098   34.6804   49.9623       NaN       NaN   89.9546   41.1524        NaN
1   69.9109    9.5542    6.4364   31.2878   37.9665       NaN   82.0056       NaN   75.7976    91.3825
2       NaN   96.0910   59.6433   84.7104       NaN   84.1090       NaN   70.2881    1.7783     4.1177
3       NaN   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968       NaN       NaN    80.0780
4   20.5019       NaN   89.2486   67.6559   58.6359   78.2257   60.9116       NaN   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675       NaN   19.7031   34.4233   93.3995   72.2067    12.6403


The simulated DataFrame has 44 missing records spread unevenly. Passing this untreated frame to a model like LinearRegression would trigger a fatal value error.

### SimpleImputer

Fills NaNs with pre-calculated basic summary values. Options include:

- `"mean"`: Feature average. Default for numerics.
- `"median"`: 50th percentile. Highly resistant to outliers.
- `"most_frequent"`: The mode. Excellent for categorical subsets.
- `"constant"`: Developer-defined static fallback.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)

print("Learned means for each feature:")
for feat, mean_val in zip(df.columns, imputer.statistics_):
    print(f"  {feat}: {mean_val:.4f}")
print()
print("Missing values after imputation:", int(imputed_df.isna().sum().sum()))
print()
print("First 6 rows after SimpleImputer (mean):")
print(imputed_df.head(6).to_string())

Learned means for each feature:
  Feature1: 52.5586
  Feature2: 53.8647
  Feature3: 52.8975
  Feature4: 54.9616
  Feature5: 46.0559
  Feature6: 57.8230
  Feature7: 51.1450
  Feature8: 50.7150
  Feature9: 60.6167
  Feature10: 48.4000

Missing values after imputation: 0

First 6 rows after SimpleImputer (mean):
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015   53.8647   42.0098   34.6804   49.9623   57.8230   51.1450   89.9546   41.1524    48.4000
1   69.9109    9.5542    6.4364   31.2878   37.9665   57.8230   82.0056   50.7150   75.7976    91.3825
2   52.5586   96.0910   59.6433   84.7104   46.0559   84.1090   51.1450   70.2881    1.7783     4.1177
3   52.5586   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968   50.7150   60.6167    80.0780
4   20.5019   53.8647   89.2486   67.6559   58.6359   78.2257   60.9116   50.7150   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675   46.0559   19.7031   34.4233

While computationally rapid, replacing values globally with a rigid mean artificially crushes a feature's variance and ignores underlying multivariate correlations. Use this mostly when data gaps are minute (under 5%).

### KNNImputer

`KNNImputer()` calculates localized replacements based on the values of the $k$ closest neighboring points, utilizing standard Euclidean distance metrics:

$$d(\mathbf{x}_i, \mathbf{x}_j) = \sqrt{\sum_{l \in \text{observed}} (x_{il} - x_{jl})^2}$$

This method allows estimations to be uniquely tailored per sample.

In [ ]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)

knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)

print("Missing values after KNN imputation:", int(knn_imputed_df.isna().sum().sum()))
print()
print("First 6 rows after KNNImputer (k=2):")
print(knn_imputed_df.head(6).to_string())
print()
f1_missing_idx = df[df['Feature1'].isna()].index.tolist()
if f1_missing_idx:
    print(f"Feature1 was NaN at indices: {f1_missing_idx}")
    print(f"  SimpleImputer filled with: {[f'{imputed_df.loc[i, "Feature1"]:.4f}' for i in f1_missing_idx]}")
    print(f"  KNNImputer filled with:    {[f'{knn_imputed_df.loc[i, "Feature1"]:.4f}' for i in f1_missing_idx]}")

Missing values after KNN imputation: 0

First 6 rows after KNNImputer (k=2):
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015   48.9540   42.0098   34.6804   49.9623   93.9104   52.5493   89.9546   41.1524    59.8337
1   69.9109    9.5542    6.4364   31.2878   37.9665   86.7935   82.0056   45.4162   75.7976    91.3825
2   67.7523   96.0910   59.6433   84.7104   73.3383   84.1090   59.0129   70.2881    1.7783     4.1177
3   47.8809   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968   64.5103   39.7268    80.0780
4   20.5019   31.6709   89.2486   67.6559   58.6359   78.2257   60.9116   25.9036   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675   59.7168   19.7031   34.4233   93.3995   72.2067    12.6403

Feature1 was NaN at indices: [2, 3, 7]
  SimpleImputer filled with: ['52.5586', '52.5586', '52.5586']
  KNNImputer filled with:    ['67.7523', '47.8809', '47.6297']


Compared to SimpleImputer, KNN actively adapts to local topological data structures. The downside is steep computational weight for large datasets, measuring roughly at $O(n^2 \cdot p)$.

### IterativeImputer

Inspired by MICE algorithms, `IterativeImputer()` resolves missing gaps dynamically by treating the target column as the dependent variable while rotating all other features as predictors.

It repeats cyclic regression loops to build deep estimates. It currently remains marked as experimental within the scikit-learn library.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(random_state=2024, max_iter=10)

iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)

print("Missing values after IterativeImputer:", int(iterative_imputed_df.isna().sum().sum()))
print()
print("First 6 rows after IterativeImputer:")
print(iterative_imputed_df.head(6).to_string())
print()
f1_missing_idx = df[df['Feature1'].isna()].index.tolist()
if f1_missing_idx:
    print(f"Comparison of imputed values for Feature1 (NaN indices: {f1_missing_idx}):")
    print(f"  SimpleImputer (mean):  {[f'{imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")
    print(f"  KNNImputer (k=2):      {[f'{knn_imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")
    print(f"  IterativeImputer:      {[f'{iterative_imputed_df.loc[i, "Feature1"]:.2f}' for i in f1_missing_idx]}")

Missing values after IterativeImputer: 0

First 6 rows after IterativeImputer:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0   58.8015   54.3650   42.0098   34.6804   49.9623   58.6806   51.1781   89.9546   41.1524    48.2923
1   69.9109    9.5542    6.4364   31.2878   37.9665   60.0706   82.0056   50.7103   75.7976    91.3825
2   52.5397   96.0910   59.6433   84.7104   46.1213   84.1090   51.1554   70.2881    1.7783     4.1177
3   52.6306   25.1767   83.7324   88.0231   16.8869   97.2056   84.6968   50.6803   50.6509    80.0780
4   20.5019   53.1990   89.2486   67.6559   58.6359   78.2257   60.9116   50.6917   65.1142    99.1192
5   10.6063   76.8254   20.0527    5.3675   46.1563   19.7031   34.4233   93.3995   72.2067    12.6403

Comparison of imputed values for Feature1 (NaN indices: [2, 3, 7]):
  SimpleImputer (mean):  ['52.56', '52.56', '52.56']
  KNNImputer (k=2):      ['67.75', '47.88', '47.63']
  IterativeImputer:      

To summarize missingness approaches:

- **<5% Missing:** Rely on `SimpleImputer` to save compute time.
- **5-10% Missing:** Transition to `KNNImputer` or `IterativeImputer` to preserve complex relationships.
- **10%+ Missing:** Consider outright column dropping or specialized heuristic tools.

We will retain the `iterative_imputed_df` table moving forward.

## Scaling Techniques

Disparate scaling disrupts gradient convergence and directly corrupts distance-based mathematical structures (like K-Means).

Main standardization and scaling components:

| Scaler | Formula | Output Bounds | Use-case |
|:---|:---|:---|:---|
| `StandardScaler` | $z = \frac{x - \mu}{\sigma}$ | Unbounded | Generally Gaussian data |
| `MinMaxScaler` | $x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$ | $[0, 1]$ | Strict bounds, neural nets |
| `Normalizer` | $x' = \frac{x}{\|\mathbf{x}\|}$ | Unit norm | Row-wise sparse adjustments |

### StandardScaler

Transforms distributions to center securely around a mean of zero, yielding unit variance. Commonly referred to as z-score calculation.

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)

print("Learned parameters:")
print(f"  Means (mu):  {np.round(scaler.mean_, 2)}")
print(f"  Stds (sigma): {np.round(scaler.scale_, 2)}")
print()
print("After scaling -- column statistics:")
print(f"  Means:  {np.round(scaled_df.mean().values, 10)}")
print(f"  Stds:   {np.round(scaled_df.std(ddof=0).values, 4)}")
print()
print("First 6 rows after StandardScaler:")
print(scaled_df.head(6).to_string())

Learned parameters:
  Means (mu):  [52.56 53.9  52.9  54.96 46.06 57.86 51.14 50.72 60.03 48.4 ]
  Stds (sigma): [22.95 23.65 29.18 22.68 20.66 25.69 21.63 23.3  24.17 30.08]

After scaling -- column statistics:
  Means:  [-0.  0.  0. -0.  0. -0. -0. -0.  0. -0.]
  Stds:   [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after StandardScaler:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0    0.2719    0.0195   -0.3731   -0.8943    0.1888    0.0318    0.0019    1.6838   -0.7809    -0.0036
1    0.7560   -1.8749   -1.5922   -1.0439   -0.3917    0.0859    1.4269   -0.0005    0.6525     1.4291
2   -0.0009    1.7835    0.2313    1.3120    0.0029    1.0216    0.0009    0.8397   -2.4099    -1.4723
3    0.0030   -1.2145    1.0568    1.4580   -1.4118    1.5313    1.5513   -0.0018   -0.3879     1.0532
4   -1.3971   -0.0298    1.2459    0.5599    0.6085    0.7926    0.4518   -0.0013    0.2105     1.6863
5   -1.8283    0.9690   -1.1255   -2.186

Post-scaling, mean outputs sit exactly at 0.0 with 1.0 standard deviations. Crucially, the scaler object archives its internal `mu` and `sigma` factors to deterministically align testing datasets downstream.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(minmax_scaled_data, columns=iterative_imputed_df.columns)

print("Learned parameters:")
print(f"  Min per feature: {np.round(minmax_scaler.data_min_, 2)}")
print(f"  Max per feature: {np.round(minmax_scaler.data_max_, 2)}")
print()
print("After scaling -- value range per column:")
print(f"  Min: {np.round(minmax_scaled_df.min().values, 4)}")
print(f"  Max: {np.round(minmax_scaled_df.max().values, 4)}")
print()
print("First 6 rows after MinMaxScaler:")
print(minmax_scaled_df.head(6).to_string())

Learned parameters:
  Min per feature: [ 1.91  9.55  6.4   5.37  6.19 10.46 17.49  0.88  1.78  4.12]
  Max per feature: [96.18 96.09 99.97 88.02 83.11 97.21 94.93 93.4  99.69 99.12]

After scaling -- value range per column:
  Min: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  Max: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after MinMaxScaler:
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0    0.6035    0.5178    0.3805    0.3546    0.5690    0.5559    0.4350    0.9628    0.4022     0.4650
1    0.7214    0.0000    0.0004    0.3136    0.4131    0.5719    0.8331    0.5386    0.7560     0.9186
2    0.5371    1.0000    0.5690    0.9599    0.5191    0.8490    0.4347    0.7502    0.0000     0.0000
3    0.5380    0.1805    0.8264    1.0000    0.1390    1.0000    0.8678    0.5383    0.4992     0.7996
4    0.1972    0.5043    0.8854    0.7536    0.6818    0.7812    0.5607    0.5384    0.6469     1.0000
5    0.0922    0.7774    0.1459    0.0000    0

### MinMaxScaler

Transforms distinct data tightly into a defined `[0, 1]` envelope.

$$x'_{ij} = \frac{x_{ij} - x_{\min,j}}{x_{\max,j} - x_{\min,j}}$$

MinMaxScaler provides rigid boundaries useful for precise network expectations but fails heavily if exposed to massive outliers, which forcibly compress the standard distribution clusters.

In [ ]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer(norm='l2')
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)

row_norms = np.sqrt((normalized_df ** 2).sum(axis=1))
print("L2 norm of each row (should all be 1.0):")
print(np.round(row_norms.values, 6))
print()
print("First 6 rows after Normalizer (L2):")
print(normalized_df.head(6).to_string())

L2 norm of each row (should all be 1.0):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

First 6 rows after Normalizer (L2):
   Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0    0.3392    0.3136    0.2423    0.2000    0.2882    0.3385    0.2952    0.5189    0.2374     0.2786
1    0.3767    0.0515    0.0347    0.1686    0.2046    0.3237    0.4419    0.2732    0.4084     0.4924
2    0.2643    0.4834    0.3001    0.4262    0.2320    0.4232    0.2574    0.3536    0.0089     0.0207
3    0.2438    0.1166    0.3878    0.4077    0.0782    0.4502    0.3923    0.2347    0.2346     0.3709
4    0.0959    0.2489    0.4175    0.3165    0.2743    0.3659    0.2849    0.2371    0.3046     0.4637
5    0.0681    0.4934    0.1288    0.0345    0.2964    0.1265    0.2211    0.5998    0.4637     0.0812


### Normalizer

Operates entirely **row-wise**. It guarantees each sample vector rests cleanly on the surface boundaries of a unit hypersphere.

$$\mathbf{x}'_i = \frac{\mathbf{x}_i}{\|\mathbf{x}_i\|}$$

Primarily leveraged for sparsified document groupings or TF-IDF natural language constraints.

## Categorical Variable Encoding

Variables like locations or internal product strings require numerical mapping schemas:

- **Nominal sets** (e.g., Apple, Banana, Orange): Implement `OneHotEncoder()` to prevent fake numerical hierarchy.
- **Ordinal subsets** (e.g., Small, Medium, Large): Leverage `OrdinalEncoder()` or `LabelEncoder()` safely.

### Simulating Categorical Datasets

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(2024)
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame({
    "Department": np.random.choice(categories, size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})

print(f"Shape: {categorical_data.shape}")
print(f"Unique values per column:")
for col in categorical_data.columns:
    print(f"  {col}: {sorted(categorical_data[col].unique())} ({categorical_data[col].nunique()} unique)")
print()
print("First 8 rows:")
print(categorical_data.head(8).to_string())

Shape: (20, 3)
Unique values per column:
  Department: ['A', 'B', 'C', 'D'] (4 unique)
  Position: ['Junior', 'Manager', 'Senior'] (3 unique)
  Location: ['CHI', 'LA', 'NY', 'SF'] (4 unique)

First 8 rows:
  Department Position Location
0          A  Manager       LA
1          C  Manager       LA
2          A   Junior       NY
3          A  Manager       LA
4          D  Manager       NY
5          A   Senior       LA
6          C  Manager       SF
7          D  Manager       LA


This sample holds text arrays lacking a unified mathematical ranking structure.

### OneHotEncoder

Transforms text states into multiple independent binary columns, negating hierarchical implications entirely.

$$\text{OHE}(x_i = c) = \begin{cases} 1 & \text{if } x_i = c \\ 0 & \text{otherwise} \end{cases}$$

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)

onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data,
    columns=onehot_encoder.get_feature_names_out()
)

print(f"Original shape: {categorical_data.shape}")
print(f"Encoded shape:  {onehot_encoded_df.shape}")
print(f"New columns: {list(onehot_encoded_df.columns)}")
print()
print("First 8 rows (showing subset of columns):")
print(onehot_encoded_df.head(8).to_string())

Original shape: (20, 3)
Encoded shape:  (20, 11)
New columns: ['Department_A', 'Department_B', 'Department_C', 'Department_D', 'Position_Junior', 'Position_Manager', 'Position_Senior', 'Location_CHI', 'Location_LA', 'Location_NY', 'Location_SF']

First 8 rows (showing subset of columns):
   Department_A  Department_B  Department_C  Department_D  Position_Junior  Position_Manager  Position_Senior  Location_CHI  Location_LA  Location_NY  Location_SF
0        1.0000        0.0000        0.0000        0.0000           0.0000            1.0000           0.0000        0.0000       1.0000       0.0000       0.0000
1        0.0000        0.0000        1.0000        0.0000           0.0000            1.0000           0.0000        0.0000       1.0000       0.0000       0.0000
2        1.0000        0.0000        0.0000        0.0000           1.0000            0.0000           0.0000        0.0000       0.0000       1.0000       0.0000
3        1.0000        0.0000        0.0000        0.0000  

OHE effectively expanded 3 parameters into 11 distinct binary paths.

However, immense categorical subsets can force explosive matrix scaling that overburdens internal memory bounds.

### LabelEncoder

Attaches direct integers to individual text variables. Highly compact, yet prone to generating destructive hierarchical illusions for standard nominal metrics.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()

for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(
        categorical_data[column]
    )
    print(f"{column} mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

print()
print("First 8 rows after LabelEncoder:")
print(label_encoded_df.head(8).to_string())

Department mapping: {'A': np.int64(0), 'B': np.int64(1), 'C': np.int64(2), 'D': np.int64(3)}
Position mapping: {'Junior': np.int64(0), 'Manager': np.int64(1), 'Senior': np.int64(2)}
Location mapping: {'CHI': np.int64(0), 'LA': np.int64(1), 'NY': np.int64(2), 'SF': np.int64(3)}

First 8 rows after LabelEncoder:
   Department_encoded  Position_encoded  Location_encoded
0                   0                 1                 1
1                   2                 1                 1
2                   0                 0                 2
3                   0                 1                 1
4                   3                 1                 2
5                   0                 2                 1
6                   2                 1                 3
7                   3                 1                 1


While computationally small, labeling non-ordinal categories can ruin linear or distance-based predictors. Reserve `LabelEncoder` explicitly for ordered strings or isolated tree models.

### ColumnTransformer

Automates specific transform applications for split data structures containing unified nominal and numerical arrays.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
import pandas as pd

np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})

print("Original mixed dataset:")
print(mixed_data.head(8).to_string())
print(f"\nDtypes:\n{mixed_data.dtypes.to_string()}")

Original mixed dataset:
   Age     Salary  Experience Department Position
0   33 59420.3600          12    Finance  Manager
1   57 82895.9200          17      Sales   Junior
2   25 38165.7600          16    Finance  Manager
3   52 38242.3600           7         IT   Junior
4   61 55088.6500           8      Sales  Manager
5   26 78688.8800           9      Sales   Senior
6   60 49585.6200          14    Finance   Junior
7   35 47992.5800          17         HR  Manager

Dtypes:
Age             int64
Salary        float64
Experience      int64
Department     object
Position       object


This dataset combines standard numerical data directly alongside two categorical formats that require alternative pipelines.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

transformed_data = column_transformer.fit_transform(mixed_data)

numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = (
    column_transformer.named_transformers_["cat"]
    .get_feature_names_out(["Department", "Position"])
)
all_cols = numeric_cols + list(categorical_cols)

transformed_df = pd.DataFrame(transformed_data, columns=all_cols)

print(f"Original shape: {mixed_data.shape}")
print(f"Transformed shape: {transformed_df.shape}")
print(f"\nColumns: {list(transformed_df.columns)}")
print(f"\nFirst 8 rows:")
print(transformed_df.head(8).to_string())

Original shape: (20, 5)
Transformed shape: (20, 10)

Columns: ['Age_scaled', 'Salary_scaled', 'Experience_scaled', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Sales', 'Position_Junior', 'Position_Manager', 'Position_Senior']

First 8 rows:
   Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager  Position_Senior
0     -1.0453        -0.3050             0.3273              1.0000         0.0000         0.0000            0.0000           0.0000            1.0000           0.0000
1      0.7277         1.1051             1.2625              0.0000         0.0000         0.0000            1.0000           1.0000            0.0000           0.0000
2     -1.6364        -1.5818             1.0754              1.0000         0.0000         0.0000            0.0000           0.0000            1.0000           0.0000
3      0.3583        -1.5772            -0.6078              0.0

In one call, `ColumnTransformer` effectively routed numerical traits to standardization formulas while simultaneously binning text matrices. Setting `remainder="passthrough"` safeguards non-targeted variables against deletion.

## Pipelines in scikit-learn

Pipelines seamlessly automate multiple processing phases, ensuring safe execution logic for advanced evaluations without risking leakage.

### Building a Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

X = transformed_df.iloc[:, :-1]
y = transformed_df.iloc[:, -1]

print(f"Feature matrix X: {X.shape}")
print(f"Target vector y:  {y.shape}")
print(f"Target unique values: {sorted(y.unique())}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Feature matrix X: (20, 9)
Target vector y:  (20,)
Target unique values: [np.float64(0.0), np.float64(1.0)]

Train size: 16, Test size: 4


You should always sever testing data early in the process before enforcing preprocessing rules to maintain evaluation integrity.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

X_train_transformed = pipeline.fit_transform(X_train)

X_test_transformed = pipeline.transform(X_test)

X_train_transformed = pd.DataFrame(
    X_train_transformed, columns=X_train.columns, index=X_train.index
)
X_test_transformed = pd.DataFrame(
    X_test_transformed, columns=X_test.columns, index=X_test.index
)

print("Pipeline steps:", [name for name, _ in pipeline.steps])
print(f"\nTraining data after pipeline (shape {X_train_transformed.shape}):")
print(X_train_transformed.head(6).to_string())
print(f"\nTest data after pipeline (shape {X_test_transformed.shape}):")
print(X_test_transformed.head().to_string())

Pipeline steps: ['imputer', 'scaler']

Training data after pipeline (shape (16, 9)):
    Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager
13      0.8343        -0.8675            -1.5598             -0.5774        -0.4804         2.0817           -0.7746          -0.5774           -1.0000
12     -0.6892         1.2828            -0.1509             -0.5774         2.0817        -0.4804           -0.7746          -0.5774            1.0000
16      0.5441        -1.1734            -1.7610             -0.5774        -0.4804         2.0817           -0.7746          -0.5774            1.0000
5      -1.3421         0.8215            -0.1509             -0.5774        -0.4804        -0.4804            1.2910          -0.5774           -1.0000
17      1.4147         0.6111             0.6541             -0.5774        -0.4804        -0.4804            1.2910          -0.5774            1.0000
2  

The operations follow an uncompromising linear sequence:

$$\underbrace{\text{train\_test\_split}(X, y)}_{\text{Step 1}} \rightarrow \underbrace{\text{pipeline.fit\_transform}(X_{\text{train}})}_{\text{Step 2}} \rightarrow \underbrace{\text{pipeline.transform}(X_{\text{test}})}_{\text{Step 3}}$$

Breeching this timeline exposes future testing sets directly to algorithm learning phases.

### Visualizing the Pipeline

scikit-learn contains integrated tools to physically map pipeline variables.

In [ ]:
from sklearn import set_config

set_config(display="diagram")

print("Pipeline structure:")
print(repr(pipeline))
print()
for i, (name, step) in enumerate(pipeline.steps):
    print(f"  Step {i+1}: '{name}' -> {type(step).__name__}()")
    params = step.get_params()
    key_params = {k: v for k, v in params.items() if v is not None and v != False}
    if key_params:
        for k, v in key_params.items():
            print(f"    {k} = {v}")

Pipeline structure:
Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

Step details:
  Step 1: 'imputer' -> SimpleImputer()
    copy = True
    missing_values = nan
    strategy = mean
  Step 2: 'scaler' -> StandardScaler()
    copy = True
    with_mean = True
    with_std = True


Jupyter naturally maps this object interactively to ensure pipeline validity.

1. Ensure complete dataset modifications are built inside pipelines.
2. Final execution steps generally terminate in estimators.
3. Adjust pipeline parameters universally utilizing standard `GridSearchCV` formats.
4. Archive objects natively using `joblib.dump()`.

## Feature Engineering

Feature engineering dictates overall algorithmic efficiency limits through:

1. **Feature extraction**: Assembling sophisticated combination mappings.
2. **Feature selection**: Trimming dead correlations to boost predictive sharpness.

### PolynomialFeatures

Linear lines rarely describe organic variables effectively. `PolynomialFeatures()` generates interactions between inputs according to a degree $d$:

$$[1,\ x_1,\ x_2,\ x_1^2,\ x_1 x_2,\ x_2^2]$$

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=True)

poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features,
    columns=poly.get_feature_names_out(X_train_transformed.columns)
)

print(f"Original features: {X_train_transformed.shape[1]}")
print(f"Polynomial features (degree=2): {poly_features_df.shape[1]}")
print(f"Feature expansion factor: {poly_features_df.shape[1] / X_train_transformed.shape[1]:.1f}x")
print()
print(f"First 5 new column names (of {poly_features_df.shape[1]}):")
for col in poly_features_df.columns[:5]:
    print(f"  {col}")
print(f"  ... and {poly_features_df.shape[1] - 5} more")
print()
print("First 4 rows (subset of columns):")
print(poly_features_df.iloc[:4, :6].to_string())

Original features: 9
Polynomial features (degree=2): 55
Feature expansion factor: 6.1x

First 5 new column names (of 55):
  1
  Age_scaled
  Salary_scaled
  Experience_scaled
  Department_Finance
  ... and 50 more

First 4 rows (subset of columns):
       1  Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR
0 1.0000      0.8343        -0.8675            -1.5598             -0.5774        -0.4804
1 1.0000     -0.6892         1.2828            -0.1509             -0.5774         2.0817
2 1.0000      0.5441        -1.1734            -1.7610             -0.5774        -0.4804
3 1.0000     -1.3421         0.8215            -0.1509             -0.5774        -0.4804


Expansion scales intensely: $\binom{p + d}{d}$. Excessive polynomials crash environments quickly and invariably necessitate rigorous feature regularization parameters to purge newly generated noise streams.

### KBinsDiscretizer

Separates loose numbers into strict organizational zones based on specific constraints (`"uniform"`, `"quantile"`, or `"kmeans"`).

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")

binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)

print(f"Bin edges per feature (first 3 features):")
for i, col in enumerate(X_train_transformed.columns[:3]):
    print(f"  {col}: {np.round(kbins.bin_edges_[i], 3)}")
print()
print("Binned values (0, 1, or 2 for 3 bins):")
print(binned_df.head(8).to_string())
print()
print("Value counts for first feature:")
print(binned_df.iloc[:, 0].value_counts().sort_index().to_string())

Bin edges per feature (first 3 features):
  Age_scaled: [-1.415 -0.472  0.472  1.415]
  Salary_scaled: [-1.48  -0.432  0.617  1.665]
  Experience_scaled: [-1.761 -0.688  0.386  1.459]

Binned values (0, 1, or 2 for 3 bins):
   Age_scaled  Salary_scaled  Experience_scaled  Department_Finance  Department_HR  Department_IT  Department_Sales  Position_Junior  Position_Manager
0      2.0000         0.0000             0.0000              0.0000         0.0000         2.0000            0.0000           0.0000            0.0000
1      0.0000         2.0000             1.0000              0.0000         2.0000         0.0000            0.0000           0.0000            2.0000
2      2.0000         0.0000             0.0000              0.0000         0.0000         2.0000            0.0000           0.0000            2.0000
3      0.0000         2.0000             1.0000              0.0000         0.0000         0.0000            2.0000           0.0000            0.0000
4      2.0000        

Bin mapping fundamentally converts linear progressions into explicit steps. This substantially boosts non-linear interpretation capacity specifically for standard linear-focused analytical configurations.

### Feature Selection with RFE

Recursive Feature Elimination isolates and abandons underperforming parameter points by actively scoring coefficients and rebuilding test arrays dynamically.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(
    estimator=LinearRegression(),
    n_features_to_select=1
)

rfe.fit(X_train_transformed, y_train)

ranking_df = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Ranking': rfe.ranking_,
    'Selected': rfe.support_
}).sort_values('Ranking')

print("RFE Feature Rankings (1 = most important):")
print(ranking_df.to_string(index=False))
print(f"\nBest feature: {ranking_df.iloc[0]['Feature']}")

RFE Feature Rankings (1 = most important):
           Feature  Ranking  Selected
  Position_Manager        1      True
   Position_Junior        2     False
     Salary_scaled        3     False
     Department_IT        4     False
        Age_scaled        5     False
     Department_HR        6     False
Department_Finance        7     False
 Experience_scaled        8     False
  Department_Sales        9     False

Best feature: Position_Manager


Highest coefficient signals lock to index $1$. Dynamic execution processes like `RFECV()` deploy these elimination matrices inherently alongside cross-validation cycles to fine-tune final array bounds.

### SelectFromModel

Executes single-cycle training loops that strip input columns strictly based on absolute numerical bounds.

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    threshold='mean'
)

selector.fit(X_train_transformed, y_train)

selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()

feature_importance = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Importance (|coef|)': np.abs(selector.estimator_.coef_),
    'Selected': selected_features_mask
}).sort_values('Importance (|coef|)', ascending=False)

print("Feature importance and selection:")
print(feature_importance.to_string(index=False))
print(f"\nThreshold (mean importance): {np.mean(np.abs(selector.estimator_.coef_)):.6f}")
print(f"Features selected: {len(selected_features)} of {X_train_transformed.shape[1]}")
print(f"Selected: {selected_features}")

Feature importance and selection:
           Feature  Importance (|coef|)  Selected
  Position_Manager               0.5000      True
   Position_Junior               0.4330      True
 Experience_scaled               0.0000     False
     Salary_scaled               0.0000     False
        Age_scaled               0.0000     False
Department_Finance               0.0000     False
     Department_HR               0.0000     False
     Department_IT               0.0000     False
  Department_Sales               0.0000     False

Threshold (mean importance): 0.103668
Features selected: 2 of 9
Selected: ['Position_Junior', 'Position_Manager']


While SelectFromModel is computationally efficient, RFE processes internal interactions with far superior accuracy. Both algorithms still pale in comparison to applying native domain knowledge to build specific formulas manually.

## Transformation Scaling Pathing

There is no universally "correct" scaler configuration. Base selections directly on core constraints:

- **Contains Outliers:** Default immediately to `RobustScaler()`.
- **Normal Spread:** Select `StandardScaler()` or standard ranges.
- **Heavily Skewed:** Utilize specific `PowerTransformer()` conversions.
- **Specific Bounds Needed:** Force `MinMaxScaler()` setups.
- **Sparse Matrix Density:** Map utilizing `Normalizer()` constraints.

## Summary Checklist

- **Missing Inputs:** Execute `SimpleImputer` or `KNNImputer` algorithms.
- **Scaling Matrices:** Format via `StandardScaler` or `Normalizer`.
- **Data Encoding:** Translate text using `OneHotEncoder` routines.
- **Feature Flow:** Consistently combine methods within closed `Pipeline` loops.
- **Architecture Limits:** Create via `PolynomialFeatures` and prune cleanly using `RFE`.

Never forget to split evaluation data prior to applying these structural data limits to prevent test poisoning.